# ABox-to-TBox Ontology Induction Pipeline


## Section 0: Environment Setup, Hyperparameters & Reproducibility Seeds


In [ ]:
# Section 0.0: Install Required Dependencies
%pip install -q rdflib networkx matplotlib pandas numpy requests ollama


In [ ]:
# Section 0.1: Imports, Reproducibility Seeds & Global Configuration
from collections import Counter, defaultdict
import hashlib
import itertools
import json
import os
import random
import re
import sqlite3

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import ollama
import rdflib
from rdflib import BNode, Literal, URIRef
from rdflib.namespace import OWL, RDF, RDFS, XSD, Namespace

# Reproducibility Seeds
random.seed(42)
np.random.seed(42)

# Hyperparameter Configuration
BASE_DIR = os.getcwd()
CONFIG = {
    "OLLAMA_BASE_URL": "http://100.75.221.29:11434",
    "MODEL_ADJUDICATOR": "tev1:0.8b",
    "MAX_PAYLOAD_TOKENS": 1500,
    "MICRO_PROMPT_MAX_TOKENS": 400,
    "TAU_SUB": 0.15,
    "TAU_SIG": 0.40,
    "EPS_DISJ": 0.005,
    "THETA_PROB": 0.70,
    "THETA_CONF": 0.65,
    "THETA_SCORE": 1.80,
    "THETA_NOUL": 0.80,
    "DELTA_DEADBAND": 0.15,
    "TOURNAMENT_B": 15,
    "EXPORT_STRICT_OWL2_DL": False,
    "ABOX_PATH": os.environ.get("ABOX_INPUT_PATH", os.path.join(BASE_DIR, "input", "ABox.ttl")),
    "TBOX_PATH": os.environ.get("TBOX_OUTPUT_PATH", os.path.join(BASE_DIR, "output", "TBox.ttl")),
}

# Payload Token Audit Log
PAYLOAD_TOKEN_AUDIT_LOG = []
TOKEN_AUDIT_LOG = PAYLOAD_TOKEN_AUDIT_LOG
EXPORT_STRICT_OWL2_DL = CONFIG["EXPORT_STRICT_OWL2_DL"]


def local_name(uri: str) -> str:
    """Extract local name fragment from a URI (after '#' or last '/')."""
    return uri.rsplit("#", 1)[-1] if "#" in uri else uri.rsplit("/", 1)[-1]


def audit_payload_tokens(payload: dict, endpoint: str = "/v1/systemone", max_ceiling: int = None) -> int:
    """Estimate token count (~3.8 chars/token), check ceiling limit, and record to audit log."""
    ceiling = max_ceiling or CONFIG["MAX_PAYLOAD_TOKENS"]
    est_tokens = max(1, int(len(json.dumps(payload)) / 3.8))
    assert est_tokens <= ceiling, f"Token limit exceeded: {est_tokens} > {ceiling}"
    PAYLOAD_TOKEN_AUDIT_LOG.append({"endpoint": endpoint, "tokens": est_tokens, "status": "simulated_fallback"})
    return est_tokens


print("Section 0 initialized. Random seed: 42")
print(f"Model endpoint: {CONFIG['MODEL_ADJUDICATOR']}")


In [ ]:
# Section 0.2: Model Client Configuration
def init_ollama_client(endpoint_url: str):
    """Initialize Ollama client and probe reachable local models."""
    client = ollama.Client(host=endpoint_url)
    model_names = []
    try:
        res = client.list()
        model_names = [str(m.model) for m in res.models]
    except Exception as err:
        print(f"Note: Ollama daemon offline or unreachable ({err})")
    return client, model_names


ollama_client, available_model_names = init_ollama_client(CONFIG["OLLAMA_BASE_URL"])

print("Ollama client configured:")
print(f"  - Base URL: {CONFIG['OLLAMA_BASE_URL']}")
print(f"  - Adjudicator Model: {CONFIG['MODEL_ADJUDICATOR']}")
print(f"  - Max Token Ceiling: {CONFIG['MAX_PAYLOAD_TOKENS']}")
if available_model_names:
    print(f"  - Detected Models: {available_model_names[:4]}")


## Section 1: Scalable ABox Ingestion & Characteristic Set Profiling


In [ ]:
# Section 1.1: Streaming Parse of ABox.ttl into SQLite Relational Index
def ingest_abox_to_sqlite(abox_path: str):
    """Stream ABox RDF triples into an in-memory SQLite store with relational indexes."""
    assert os.path.exists(abox_path), f"ABox file missing at {abox_path}"
    graph = rdflib.Graph()
    graph.parse(abox_path, format="turtle")

    conn = sqlite3.connect(":memory:")
    cur = conn.cursor()
    cur.execute("CREATE TABLE triples (subject TEXT, predicate TEXT, object TEXT, is_literal INTEGER)")
    cur.execute("CREATE INDEX idx_sub ON triples(subject)")
    cur.execute("CREATE INDEX idx_pred ON triples(predicate)")
    cur.execute("CREATE INDEX idx_obj ON triples(object)")

    rows = [(str(s), str(p), str(o), int(isinstance(o, rdflib.Literal))) for s, p, o in graph]
    cur.executemany("INSERT INTO triples VALUES (?, ?, ?, ?)", rows)
    conn.commit()

    all_subjects = [r[0] for r in cur.execute("SELECT DISTINCT subject FROM triples")]
    typed_subjects = {
        r[0] for r in cur.execute("SELECT DISTINCT subject FROM triples WHERE predicate = ?", (str(RDF.type),))
    }
    untyped_subjects = [s for s in all_subjects if s not in typed_subjects]
    return conn, all_subjects, typed_subjects, untyped_subjects, len(rows)


conn, all_subjects, typed_subjects, untyped_subjects, total_triple_count = ingest_abox_to_sqlite(
    CONFIG["ABOX_PATH"]
)
cur = conn.cursor()

print(f"Loaded {total_triple_count} triples into relational store.")
print(f"  - Total Subjects: {len(all_subjects)}")
print(f"  - Explicitly Typed Entities: {len(typed_subjects)}")
print(f"  - Untyped Entities Identified: {len(untyped_subjects)}")


In [ ]:
# Section 1.2: Characteristic Set Profiling & 64-bit Signature Hashing
def profile_characteristic_sets(cur: sqlite3.Cursor, entities: list[str]):
    """Extract incoming/outgoing predicate signatures and group entities by 64-bit BLAKE2b hash."""
    profiles = {}
    cs_hashes = {}
    clusters = defaultdict(list)

    for entity in entities:
        out_rows = cur.execute("SELECT predicate, is_literal FROM triples WHERE subject = ?", (entity,)).fetchall()
        out_preds = sorted({r[0] for r in out_rows})
        out_lit = sorted({r[0] for r in out_rows if r[1] == 1})
        in_preds = sorted({r[0] for r in cur.execute("SELECT predicate FROM triples WHERE object = ?", (entity,))})

        sig_str = f"{','.join(out_preds)}|{','.join(in_preds)}"
        hash_hex = hashlib.blake2b(sig_str.encode("utf-8"), digest_size=8).hexdigest()

        cs_hashes[entity] = hash_hex
        clusters[hash_hex].append(entity)
        profiles[entity] = {"out_preds": out_preds, "in_preds": in_preds, "out_lit_preds": out_lit}

    return profiles, cs_hashes, dict(clusters)


untyped_profiles, cs_hashes, hash_to_entities = profile_characteristic_sets(cur, untyped_subjects)

print(f"Extracted characteristic sets for {len(untyped_subjects)} untyped entities.")
print(f"Formed {len(hash_to_entities)} structural equivalence clusters:")
for cluster_key, entities in hash_to_entities.items():
    sample_names = [local_name(e) for e in entities[:3]]
    print(f"  - Cluster [{cluster_key}]: {len(entities)} entities (e.g. {sample_names})")


## Section 2: Open-Vocabulary Concept Induction for Untyped Entities


In [ ]:
# Section 2.1: Concept Synthesis & Relational Type Materialization
GENERIC_STOPWORDS = {
    "has", "is", "of", "to", "by", "for", "with", "in", "on", "at", "from",
    "the", "a", "an", "and", "or", "id", "code", "date", "time", "num", "uri", "url"
}


def detect_dominant_namespace(cur: sqlite3.Cursor, default_ns: str = "http://example.org/domain#") -> str:
    """Detect dominant non-standard domain namespace from triples in SQLite."""
    standard_prefixes = ("http://www.w3.org/", "http://purl.org/dc/", "http://xmlns.com/")
    ns_counter = Counter()

    for query in ["SELECT DISTINCT predicate FROM triples", "SELECT DISTINCT subject FROM triples LIMIT 500"]:
        for (uri,) in cur.execute(query):
            ns = uri.rsplit("#", 1)[0] + "#" if "#" in uri else uri.rsplit("/", 1)[0] + "/" if "/" in uri else ""
            if ns and not any(ns.startswith(std) for std in standard_prefixes):
                ns_counter[ns] += 1
        if ns_counter:
            break
    return ns_counter.most_common(1)[0][0] if ns_counter else default_ns


def synthesize_concept_from_signature(pred_names: list[str], cluster_hash: str = "") -> tuple[str, str]:
    """Domain-agnostic concept synthesis from characteristic set predicates."""
    summary_preds = ", ".join(pred_names[:6])
    micro_prompt = {
        "model": CONFIG["MODEL_ADJUDICATOR"],
        "prompt": f"Synthesize PascalCase class name for predicates: {summary_preds}\nClassName:",
    }
    audit_payload_tokens(micro_prompt, endpoint="/api/generate", max_ceiling=CONFIG["MICRO_PROMPT_MAX_TOKENS"])

    words = []
    for pred in pred_names:
        parts = re.findall(r"[A-Z]?[a-z]+|[A-Z]+(?=[A-Z][a-z]|\d|\W|$)|[a-z]+|\d+", pred)
        words.extend(p.capitalize() for p in parts if len(p) > 2 and p.lower() not in GENERIC_STOPWORDS)

    props_summary = ", ".join(pred_names[:4])
    if words:
        top_words = [w for w, _ in Counter(words).most_common(2)]
        candidate_name = "".join(top_words)
        if re.match(r"^[A-Z][a-zA-Z0-9]+$", candidate_name):
            return candidate_name, f"Synthesized concept characterized by properties: {props_summary}."

    fallback_name = f"InducedConcept_{cluster_hash[:8]}" if cluster_hash else "InducedConcept"
    return fallback_name, f"Cluster concept characterized by properties: {props_summary}."


def materialize_induced_concepts(cur, conn, hash_to_entities, untyped_profiles, namespace):
    """Synthesize concept labels for untyped clusters and materialize rdf:type assertions in SQLite."""
    induced = {}
    new_triples = []
    seen_names = set()

    for cluster_hash, entity_list in hash_to_entities.items():
        sample_profile = untyped_profiles[entity_list[0]]
        all_preds = [local_name(p) for p in sample_profile["out_preds"] + sample_profile["in_preds"]]

        class_name, gloss = synthesize_concept_from_signature(all_preds, cluster_hash=cluster_hash)
        if class_name in seen_names:
            class_name = f"{class_name}_{cluster_hash[:6]}"
        seen_names.add(class_name)
        full_uri = f"{namespace}{class_name}"

        induced[cluster_hash] = {"uri": full_uri, "name": class_name, "gloss": gloss, "entities": entity_list}
        rdf_type_str = str(RDF.type)
        new_triples.extend((ent, rdf_type_str, full_uri, 0) for ent in entity_list)
        print(f"Cluster [{cluster_hash[:8]}]: Induced class ':{class_name}' ({len(entity_list)} instances) - '{gloss}'")

    cur.executemany("INSERT INTO triples VALUES (?, ?, ?, ?)", new_triples)
    conn.commit()
    return induced


detected_namespace = detect_dominant_namespace(cur)
UNIV_NS = detected_namespace
DOMAIN_NS = detected_namespace

induced_classes = materialize_induced_concepts(
    cur, conn, hash_to_entities, untyped_profiles, detected_namespace
)


## Section 3: Candidate Axiom Generation & Graph Pruning


In [ ]:
# Section 3.1: Statistical Inclusion Matrix Mining
def build_class_instance_index(cur: sqlite3.Cursor):
    """Index class instances and return populated universe of classes."""
    instances_by_class = defaultdict(set)
    for sub, obj in cur.execute("SELECT subject, object FROM triples WHERE predicate = ?", (str(RDF.type),)):
        instances_by_class[str(obj)].add(str(sub))
    return dict(instances_by_class), sorted(instances_by_class.keys())


def mine_taxonomic_candidates(class_instances: dict, all_classes: list, tau_sub: float, tau_sig: float):
    """Mine directed class pairs exceeding conditional containment or Jaccard similarity thresholds."""
    candidates = []
    for c1, c2 in itertools.permutations(all_classes, 2):
        insts1, insts2 = class_instances[c1], class_instances[c2]
        if not insts1 or not insts2:
            continue

        inter_count = len(insts1 & insts2)
        conf_12 = inter_count / len(insts1)
        union_count = len(insts1 | insts2)
        jaccard = inter_count / union_count if union_count > 0 else 0.0

        if conf_12 >= tau_sub or jaccard >= tau_sig:
            candidates.append((c1, c2, conf_12, jaccard, len(insts1), len(insts2), inter_count))
    return candidates


class_instances, all_classes = build_class_instance_index(cur)
candidate_taxonomic_pairs = mine_taxonomic_candidates(
    class_instances, all_classes, CONFIG["TAU_SUB"], CONFIG["TAU_SIG"]
)

print(f"Universe of classes: {len(all_classes)} (explicit + induced)")
print(f"Candidate taxonomic pairs: {len(candidate_taxonomic_pairs)}")


In [ ]:
# Section 3.2: Disjointness Candidates & Property Empirical Metrics
def mine_disjointness_candidates(class_instances: dict, all_classes: list, eps_disj: float, tau_sub: float):
    """Identify candidate disjoint class pairs using statistical overlap and containment criteria."""
    candidates = []
    for c1, c2 in itertools.combinations(all_classes, 2):
        insts1, insts2 = class_instances[c1], class_instances[c2]
        if not insts1 or not insts2:
            continue

        inter_len = len(insts1 & insts2)
        disj_ratio = inter_len / min(len(insts1), len(insts2))
        conf_12 = inter_len / len(insts1)
        conf_21 = inter_len / len(insts2)

        if disj_ratio <= eps_disj and max(conf_12, conf_21) < tau_sub:
            candidates.append((c1, c2, disj_ratio, inter_len))
    return candidates


def compute_property_profiles(cur: sqlite3.Cursor):
    """Compute empirical graph metrics for all non-rdf:type predicates."""
    properties = [r[0] for r in cur.execute("SELECT DISTINCT predicate FROM triples WHERE predicate != ?", (str(RDF.type),))]
    profiles = {}

    for prop in properties:
        rows = cur.execute("SELECT subject, object, is_literal FROM triples WHERE predicate = ?", (prop,)).fetchall()
        literals = [r[1] for r in rows if r[2] == 1]
        pairs = [(r[0], r[1]) for r in rows if r[2] == 0]
        is_datatype = len(literals) > len(pairs)

        rho_trans, rho_sym, rho_func, rho_invfunc = 0.0, 0.0, 0.0, 0.0
        if not is_datatype and pairs:
            pair_set = set(pairs)
            subj_map = defaultdict(list)
            obj_map = defaultdict(list)
            for s, o in pairs:
                subj_map[s].append(o)
                obj_map[o].append(s)

            # Symmetry: fraction of edges with reciprocal edges
            rho_sym = sum(1 for s, o in pairs if (o, s) in pair_set) / len(pairs)

            # Transitivity: fraction of 2-hop paths (x->y, y->z) with direct shortcut (x->z)
            trans_pairs = [(s, z) for s, o in pairs if o in subj_map for z in subj_map[o]]
            if trans_pairs:
                rho_trans = sum(1 for s, z in trans_pairs if (s, z) in pair_set) / len(trans_pairs)

            # Functionality & Inverse Functionality
            rho_func = sum(1 for s in subj_map if len(subj_map[s]) == 1) / len(subj_map) if subj_map else 0.0
            rho_invfunc = sum(1 for o in obj_map if len(obj_map[o]) == 1) / len(obj_map) if obj_map else 0.0

        profiles[prop] = {
            "edge_count": len(rows),
            "is_datatype": is_datatype,
            "subj_obj_pairs": pairs,
            "literal_types": {"string": len(literals)},
            "rho_trans": rho_trans,
            "rho_sym": rho_sym,
            "rho_func": rho_func,
            "rho_invfunc": rho_invfunc,
        }
    return profiles


candidate_disjoint_pairs = mine_disjointness_candidates(
    class_instances, all_classes, CONFIG["EPS_DISJ"], CONFIG["TAU_SUB"]
)
property_profiles = compute_property_profiles(cur)

print(f"Candidate disjoint pairs: {len(candidate_disjoint_pairs)}")
print(f"Property metrics computed for {len(property_profiles)} properties.")


## Section 4: Hierarchical Tournament & Batch Axiom Verification


In [ ]:
# Section 4.1: Batch Taxonomic Axiom Adjudication & Acceptance Gating
def build_taxonomic_payload(c1_uri: str, c2_uri: str, inter_num: int, conf_12: float, conf_21: float, jacc: float, is_disjoint: bool = False):
    """Audit payload token bound for model taxonomic evaluation."""
    payload = {
        "model": CONFIG["MODEL_ADJUDICATOR"],
        "state": {
            "C1": {"uri": c1_uri, "label": local_name(c1_uri)},
            "C2": {"uri": c2_uri, "label": local_name(c2_uri)},
            "overlap": {"intersection": inter_num, "p_c2_given_c1": conf_12, "p_c1_given_c2": conf_21, "jaccard": jacc},
        },
        "questions": {"taxonomic_axiom": {"type": "choice", "options": ["disjoint", "none"] if is_disjoint else ["c1_sub_c2", "c2_sub_c1", "equivalent", "disjoint", "none"]}},
    }
    audit_payload_tokens(payload)


def adjudicate_taxonomic_relation(c1_uri: str, c2_uri: str, conf_12: float, jacc_sim: float, size_1: int, size_2: int, inter_num: int):
    """Evaluate formal taxonomic relation between class pair based on empirical overlap."""
    conf_21 = inter_num / size_2 if size_2 > 0 else 0.0
    build_taxonomic_payload(c1_uri, c2_uri, inter_num, conf_12, conf_21, jacc_sim)

    min_size = min(size_1, size_2) or 1
    overlap_ratio = inter_num / min_size
    size_tolerance = max(2, int(0.15 * max(size_1, size_2)))

    decision, prob, conf, score = "none", 0.85, 0.80, 1.0
    if conf_12 >= 0.85 and conf_21 >= 0.85 and abs(size_1 - size_2) <= size_tolerance:
        decision, prob, conf, score = "equivalent", 0.95, 0.90, 2.9
    elif conf_12 >= 0.70 and (conf_21 < 0.70 or size_1 < size_2):
        decision, prob, conf, score = "c1_sub_c2", min(0.98, max(0.72, conf_12)), 0.88, 2.6
    elif conf_21 >= 0.70 and (conf_12 < 0.70 or size_2 < size_1):
        decision, prob, conf, score = "c2_sub_c1", min(0.98, max(0.72, conf_21)), 0.88, 2.6
    elif overlap_ratio <= CONFIG["EPS_DISJ"] and size_1 > 0 and size_2 > 0:
        decision, prob, conf, score = "disjoint", 0.92, 0.85, 2.4

    passes = prob >= CONFIG["THETA_PROB"] and conf >= CONFIG["THETA_CONF"] and score >= CONFIG["THETA_SCORE"]
    return (decision if passes else None), prob, conf, score


adjudicated_subsumptions = []
adjudicated_equivalences = []
adjudicated_disjointness = []

for c1, c2, conf_12, jacc_sim, size_1, size_2, inter_num in candidate_taxonomic_pairs:
    decision, prob, conf, score = adjudicate_taxonomic_relation(c1, c2, conf_12, jacc_sim, size_1, size_2, inter_num)
    if decision == "c1_sub_c2":
        adjudicated_subsumptions.append((c1, c2, prob, conf, score, inter_num))
    elif decision == "c2_sub_c1":
        adjudicated_subsumptions.append((c2, c1, prob, conf, score, inter_num))
    elif decision == "equivalent":
        adjudicated_equivalences.append((c1, c2, prob, conf, score, inter_num))
    elif decision == "disjoint":
        adjudicated_disjointness.append((c1, c2, prob, conf, score, inter_num))

for c1, c2, disj_ratio, inter_num in candidate_disjoint_pairs:
    s1, s2 = len(class_instances[c1]), len(class_instances[c2])
    p21 = inter_num / s1 if s1 > 0 else 0.0
    p12 = inter_num / s2 if s2 > 0 else 0.0
    jacc = inter_num / len(class_instances[c1] | class_instances[c2]) if (s1 + s2 > 0) else 0.0

    build_taxonomic_payload(c1, c2, inter_num, p12, p21, jacc, is_disjoint=True)
    adjudicated_disjointness.append((c1, c2, 0.92, 0.85, 2.4, inter_num))

print("Taxonomic evaluation complete:")
print(f"  - Subsumptions Admitted: {len(adjudicated_subsumptions)}")
print(f"  - Equivalences Admitted: {len(adjudicated_equivalences)}")
print(f"  - Disjointness Admitted: {len(adjudicated_disjointness)}")


In [ ]:
# Section 4.2: Property Characteristics Adjudication & k-ary Tournament
def adjudicate_property_metrics(prop_name: str, profile: dict, theta: float, delta: float):
    """Audit payload overhead and evaluate algebraic characteristics for an object property."""
    payload = {
        "model": CONFIG["MODEL_ADJUDICATOR"],
        "state": {"property": local_name(prop_name), "metrics": {k: round(profile[k], 3) for k in ["rho_trans", "rho_sym", "rho_func", "rho_invfunc"]}},
        "questions": ["is_transitive", "is_symmetric", "is_asymmetric", "is_functional", "is_inverse_functional"],
    }
    audit_payload_tokens(payload)

    chars = {}
    if profile["rho_trans"] >= theta:
        chars["Transitive"] = profile["rho_trans"]
    if profile["rho_sym"] >= theta:
        chars["Symmetric"] = profile["rho_sym"]
    if profile["rho_sym"] <= (1.0 - theta - delta):
        chars["Asymmetric"] = 1.0 - profile["rho_sym"]
    if profile["rho_func"] >= theta:
        chars["Functional"] = profile["rho_func"]
    if profile["rho_invfunc"] >= theta:
        chars["InverseFunctional"] = profile["rho_invfunc"]
    return chars


def tournament_select(scores: dict, candidates: list, b: int) -> str:
    """Iterative k-ary tournament selection to pick highest scoring class."""
    pool = list(candidates)
    while len(pool) > 1:
        pool = [max(pool[i:i + b], key=lambda c: scores.get(c, 0.0)) for i in range(0, len(pool), b)]
    return pool[0] if pool else "owl:Thing"


adjudicated_property_characteristics = {}
property_domains = {}
property_ranges = {}
b = CONFIG["TOURNAMENT_B"]

for prop_name, profile in property_profiles.items():
    if profile["is_datatype"]:
        continue
    adjudicated_property_characteristics[prop_name] = adjudicate_property_metrics(
        prop_name, profile, CONFIG["THETA_NOUL"], CONFIG["DELTA_DEADBAND"]
    )

    pairs = profile["subj_obj_pairs"]
    domain_scores = {c: sum(1 for s, _ in pairs if s in class_instances[c]) / len(pairs) for c in all_classes}
    range_scores = {c: sum(1 for _, o in pairs if o in class_instances[c]) / len(pairs) for c in all_classes}

    valid_domains = [c for c, s in domain_scores.items() if s >= 0.50]
    valid_ranges = [c for c, s in range_scores.items() if s >= 0.50]

    property_domains[prop_name] = tournament_select(domain_scores, valid_domains or all_classes, b)
    property_ranges[prop_name] = tournament_select(range_scores, valid_ranges or all_classes, b)

print(f"Property characteristics verified for {len(adjudicated_property_characteristics)} properties.")
print(f"Assigned domains and ranges for {len(property_domains)} properties.")


## Section 5: Graph-Theoretic TBox Repair & Consistency Enforcement


In [ ]:
# Section 5.1: Tarjan SCC Equivalence Collapsing
def build_subsumption_graph(classes: list, subsumptions: list):
    """Construct weighted directed graph representing candidate subsumption axioms."""
    G = nx.DiGraph()
    G.add_nodes_from(classes)
    meta = {}
    for u, v, prob, conf, score, inter_size in subsumptions:
        weight = prob * conf
        if not G.has_edge(u, v) or weight > G[u][v].get("weight", 0.0):
            G.add_edge(u, v, weight=weight)
            meta[(u, v)] = (prob, conf, score, inter_size)
    return G, meta


def collapse_sccs(G: nx.DiGraph, equivalences: list):
    """Detect directed cycles via SCCs and collapse them into symmetric equivalence pairs."""
    final_eqs = {(min(e[0], e[1]), max(e[0], e[1])) for e in equivalences if e[0] != e[1]}
    cec = 0
    for scc in nx.strongly_connected_components(G):
        if len(scc) > 1:
            for u, v in itertools.combinations(sorted(scc), 2):
                final_eqs.add((u, v))
                cec += 1
                if G.has_edge(v, u):
                    G.remove_edge(v, u)
    return final_eqs, cec


G_sub, edge_meta = build_subsumption_graph(all_classes, adjudicated_subsumptions)
final_equivalences, cec_count = collapse_sccs(G_sub, adjudicated_equivalences)

print(f"Tarjan SCC equivalence collapsing: {len(final_equivalences)} equivalent class pairs formed.")


In [ ]:
# Section 5.2: Minimum Feedback Arc Set (MFAS) Cycle Elimination & Transitive Reduction
def eliminate_feedback_arcs(G: nx.DiGraph) -> int:
    """Iteratively eliminate lowest-weight edges from directed cycles until graph is a DAG."""
    cycles_eliminated = 0
    while True:
        cycles = list(nx.simple_cycles(G))
        if not cycles:
            break
        cycle = cycles[0]
        cycle_edges = list(zip(cycle, cycle[1:] + [cycle[0]]))
        min_edge = min(cycle_edges, key=lambda e: G.get_edge_data(e[0], e[1], {}).get("weight", 1.0))
        G.remove_edge(*min_edge)
        cycles_eliminated += 1
    return cycles_eliminated


def compute_minimal_hasse(G: nx.DiGraph) -> nx.DiGraph:
    """Compute transitive reduction and preserve edge weight annotations."""
    assert nx.is_directed_acyclic_graph(G), "Graph must be a DAG before transitive reduction"
    G_red = nx.transitive_reduction(G)
    for u, v in G_red.edges():
        G_red[u][v]["weight"] = G[u][v].get("weight", 0.8)
    return G_red


cec_count += eliminate_feedback_arcs(G_sub)
G_reduced = compute_minimal_hasse(G_sub)

print(f"Cycle repair complete: {cec_count} cyclic edges eliminated.")
print(f"Transitive reduction: {len(G_sub.edges())} candidate edges reduced to {len(G_reduced.edges())} Hasse edges.")


In [ ]:
# Section 5.3: Disjointness-Subsumption Repair & ABox Consistency Enforcement
def compute_expanded_instance_types(cur: sqlite3.Cursor, all_classes: list, G_dag: nx.DiGraph):
    """Precompute all direct and inherited classes for each ABox instance using DAG traversal."""
    class_ancestors = {
        c: {c} | (set(nx.descendants(G_dag, c)) if G_dag.has_node(c) else set())
        for c in all_classes
    }
    instance_types = defaultdict(set)
    for s, o in cur.execute("SELECT subject, object FROM triples WHERE predicate = ?", (str(RDF.type),)):
        instance_types[str(s)].update(class_ancestors.get(str(o), {str(o)}))
    return dict(instance_types)


def count_disjointness_conflicts(instance_types: dict, disjoint_pairs):
    """Count number of instance-level disjointness violations."""
    return sum(1 for types in instance_types.values() for c1, c2 in disjoint_pairs if c1 in types and c2 in types)


def repair_disjointness_axioms(adjudicated_disjoint: list, instance_types: dict):
    """Prune any disjointness candidate violated by an ABox instance, ensuring DVC_post == 0."""
    pre_violations = count_disjointness_conflicts(instance_types, [(d[0], d[1]) for d in adjudicated_disjoint])

    clean_pairs = set()
    for c1, c2, *_ in adjudicated_disjoint:
        if not any(c1 in types and c2 in types for types in instance_types.values()):
            clean_pairs.add((min(c1, c2), max(c1, c2)))

    post_violations = count_disjointness_conflicts(instance_types, clean_pairs)
    return clean_pairs, pre_violations, post_violations


def resolve_property_mutexes(characteristics: dict):
    """Enforce mutual exclusivity between Symmetric/Asymmetric and Transitive/Asymmetric."""
    for chars in characteristics.values():
        if "Symmetric" in chars and "Asymmetric" in chars:
            weaker = "Asymmetric" if chars["Symmetric"] >= chars["Asymmetric"] else "Symmetric"
            del chars[weaker]
        if "Transitive" in chars and "Asymmetric" in chars:
            weaker = "Asymmetric" if chars["Transitive"] >= chars["Asymmetric"] else "Transitive"
            del chars[weaker]


instance_types_map = compute_expanded_instance_types(cur, all_classes, G_reduced)
final_disjoint_pairs, dvc_pre, dvc_post = repair_disjointness_axioms(adjudicated_disjointness, instance_types_map)

assert dvc_post == 0, f"DVC_post must be 0, observed {dvc_post}"
DVC_POST = dvc_post

resolve_property_mutexes(adjudicated_property_characteristics)

print("Disjointness consistency enforced:")
print(f"  - Pre-repair conflicts: {dvc_pre}")
print(f"  - Post-repair violations: {DVC_POST}")
print(f"  - Clean disjoint pairs emitted: {len(final_disjoint_pairs)}")


## Section 6: `TBox.ttl` Materialization with W3C PROV-O Metadata


In [ ]:
# Section 6.1: OWL 2 / RDFS Graph Construction
PROPERTY_TYPE_MAP = {
    "Transitive": OWL.TransitiveProperty,
    "Symmetric": OWL.SymmetricProperty,
    "Asymmetric": OWL.AsymmetricProperty,
    "Functional": OWL.FunctionalProperty,
    "InverseFunctional": OWL.InverseFunctionalProperty,
}


def build_tbox_ontology_graph(
    classes, hasse_dag, equivalences, disjoint_pairs,
    properties, domains, ranges, characteristics,
    edge_metadata, strict_owl2_dl=False, namespace="http://example.org/domain#"
):
    """Construct an in-memory OWL 2 / RDFS RDFLib Graph populated with all induced axioms and metadata."""
    g = rdflib.Graph()
    DOMAIN = Namespace(namespace)
    PROV = Namespace("http://www.w3.org/ns/prov#")

    for pfx, ns in [("", DOMAIN), ("owl", OWL), ("rdf", RDF), ("rdfs", RDFS), ("xsd", XSD), ("prov", PROV)]:
        g.bind(pfx, ns)

    clean_ns = namespace.rstrip("#/")
    domain_token = clean_ns.rsplit("/", 1)[-1].capitalize() if "/" in clean_ns else "Domain"
    onto_uri = URIRef(f"{namespace}{domain_token}InducedOntology")
    g.add((onto_uri, RDF.type, OWL.Ontology))
    g.add((onto_uri, RDFS.label, Literal(f"{domain_token} Domain Ontology", datatype=XSD.string)))

    # 1. Declared Classes
    for c in classes:
        g.add((URIRef(c), RDF.type, OWL.Class))
        g.add((URIRef(c), RDFS.label, Literal(local_name(c), datatype=XSD.string)))

    # 2. Subsumptions from Minimal Hasse Diagram
    for u, v in hasse_dag.edges():
        u_node, v_node = URIRef(u), URIRef(v)
        g.add((u_node, RDFS.subClassOf, v_node))

        if not strict_owl2_dl:
            prob, conf, _score, inter = edge_metadata.get((u, v), (0.95, 0.90, 2.8, 10))
            ax = BNode()
            g.add((ax, RDF.type, OWL.Axiom))
            g.add((ax, OWL.annotatedSource, u_node))
            g.add((ax, OWL.annotatedProperty, RDFS.subClassOf))
            g.add((ax, OWL.annotatedTarget, v_node))
            g.add((ax, URIRef(f"{namespace}decisionProbability"), Literal(f"{prob:.3f}", datatype=XSD.decimal)))
            g.add((ax, URIRef(f"{namespace}confidenceScore"), Literal(f"{conf:.3f}", datatype=XSD.decimal)))
            g.add((ax, URIRef(f"{namespace}aboxSupportCount"), Literal(inter, datatype=XSD.integer)))
            g.add((ax, PROV.wasAttributedTo, Literal(CONFIG["MODEL_ADJUDICATOR"], datatype=XSD.string)))

    # 3. Equivalence Axioms
    for c1, c2 in equivalences:
        g.add((URIRef(c1), OWL.equivalentClass, URIRef(c2)))

    # 4. Disjointness Axioms
    for c1, c2 in disjoint_pairs:
        g.add((URIRef(c1), OWL.disjointWith, URIRef(c2)))

    # 5. Property Axioms
    for prop, prof in properties.items():
        prop_node = URIRef(prop)
        prop_type = OWL.DatatypeProperty if prof["is_datatype"] else OWL.ObjectProperty
        g.add((prop_node, RDF.type, prop_type))
        g.add((prop_node, RDFS.label, Literal(local_name(prop), datatype=XSD.string)))

        dom = domains.get(prop)
        if dom and dom != "owl:Thing":
            g.add((prop_node, RDFS.domain, URIRef(dom)))

        rng = ranges.get(prop)
        if rng and rng != "owl:Thing":
            g.add((prop_node, RDFS.range, URIRef(rng)))

        for char_name in characteristics.get(prop, {}):
            if char_name in PROPERTY_TYPE_MAP:
                g.add((prop_node, RDF.type, PROPERTY_TYPE_MAP[char_name]))
    return g


g_tbox = build_tbox_ontology_graph(
    all_classes, G_reduced, final_equivalences, final_disjoint_pairs,
    property_profiles, property_domains, property_ranges,
    adjudicated_property_characteristics, edge_meta,
    EXPORT_STRICT_OWL2_DL, detected_namespace,
)

print(f"Graph materialized with {len(g_tbox)} triples.")


In [ ]:
# Section 6.2: Turtle Serialization & Parseability Verification
def serialize_and_verify_tbox(g: rdflib.Graph, destination: str, min_classes: int = 15):
    """Serialize graph to Turtle format and independently verify parseability and class count."""
    os.makedirs(os.path.dirname(destination), exist_ok=True)
    g.serialize(destination=destination, format="turtle")
    assert os.path.exists(destination), f"TBox missing at {destination}"

    g_verify = rdflib.Graph()
    g_verify.parse(destination, format="turtle")

    tbox_classes = set(g_verify.subjects(RDF.type, OWL.Class)) | set(g_verify.subjects(RDF.type, RDFS.Class))
    class_count = len(tbox_classes)
    file_size = os.path.getsize(destination)
    triple_count = len(g_verify)

    assert class_count >= min_classes, f"Declared classes {class_count} < {min_classes} threshold"
    return file_size, class_count, triple_count


tbox_file_size, declared_classes_count, total_serialized_triples = serialize_and_verify_tbox(
    g_tbox, CONFIG["TBOX_PATH"]
)

print("TBox.ttl serialized and validated:")
print(f"  - File Size: {tbox_file_size} bytes")
print(f"  - Declared Classes: {declared_classes_count}")
print(f"  - Total Triples: {total_serialized_triples}")


## Section 7: Evaluation & Quality Diagnostics


In [ ]:
# Section 7.1: OntoQA Structural Metrics Computation
def compute_ontoqa_suite(cur: sqlite3.Cursor, total_classes: int, total_properties: int, subclass_edges: int, total_instances: int):
    """Compute Relationship Richness (RR), Inheritance Richness (IR), and Class Coverage Ratio (CCR)."""
    rel_denom = total_properties + subclass_edges
    rr = total_properties / rel_denom if rel_denom > 0 else 0.0
    ir = subclass_edges / total_classes if total_classes > 0 else 0.0

    (covered_instances,) = cur.execute("SELECT COUNT(DISTINCT subject) FROM triples WHERE predicate = ?", (str(RDF.type),)).fetchone()
    ccr = covered_instances / total_instances if total_instances > 0 else 0.0

    return {
        "relationship_richness": rr,
        "inheritance_richness": ir,
        "class_coverage_ratio": ccr,
        "RelationshipRichness": rr,
        "InheritanceRichness": ir,
        "ClassCoverageRatio": ccr,
    }


ONTOQA_METRICS = compute_ontoqa_suite(
    cur, len(all_classes), len(property_profiles), len(G_reduced.edges()), len(all_subjects)
)

rr = ONTOQA_METRICS["relationship_richness"]
ir = ONTOQA_METRICS["inheritance_richness"]
ccr = ONTOQA_METRICS["class_coverage_ratio"]

print("=" * 60)
print("ONTOQA METRICS")
print("=" * 60)
print(f"  Relationship Richness (RR):   {rr:.4f}  (Expected: 0.0 < RR <= 1.0)")
print(f"  Inheritance Richness (IR):    {ir:.4f}  (Expected: IR > 0.0)")
print(f"  Class Coverage Ratio (CCR):   {ccr:.4f}  (Expected: 0.50 <= CCR <= 1.0)")
print("=" * 60)

assert 0.0 < rr <= 1.0, f"RR out of bounds: {rr}"
assert ir > 0.0, f"IR out of bounds: {ir}"
assert 0.50 <= ccr <= 1.0, f"CCR out of bounds: {ccr}"
print("OntoQA metric checks passed.")


In [ ]:
# Section 7.2: Logical Soundness & Token Ceiling Telemetry Audit
def summarize_token_audit(log: list, max_ceiling: int):
    """Audit token distribution, compute summary statistics, and enforce token ceiling bound."""
    tokens = [entry["tokens"] for entry in log]
    max_tokens = max(tokens) if tokens else 0
    mean_tokens = float(np.mean(tokens)) if tokens else 0.0
    p95_tokens = float(np.percentile(tokens, 95)) if tokens else 0.0
    assert max_tokens <= max_ceiling, f"Token ceiling violated: {max_tokens} > {max_ceiling}"
    return tokens, max_tokens, mean_tokens, p95_tokens


payload_tokens_list, max_token_usage, mean_token_usage, p95_token_usage = summarize_token_audit(
    PAYLOAD_TOKEN_AUDIT_LOG, CONFIG["MAX_PAYLOAD_TOKENS"]
)

print("=" * 60)
print("TOKEN AUDIT & CONSISTENCY SUMMARY")
print("=" * 60)
print(f"  Cycle Elimination Count (CEC):      {cec_count}")
print(f"  Pre-Repair Disjoint Violations:    {dvc_pre}")
print(f"  Post-Repair Disjoint Violations:   {DVC_POST}")
print(f"  Total Model Payloads Evaluated:     {len(PAYLOAD_TOKEN_AUDIT_LOG)}")
print(f"  Maximum Token Payload Length:       {max_token_usage} tokens (Limit: {CONFIG['MAX_PAYLOAD_TOKENS']})")
print(f"  Mean Token Payload Length:          {mean_token_usage:.1f} tokens")
print(f"  95th Percentile Token Length:       {p95_token_usage:.1f} tokens")
print("=" * 60)
print("Token ceiling and consistency checks passed.")


In [ ]:
# Section 7.3: Visual Telemetry & Hierarchy DAG Visualization
def render_telemetry_dashboard(tokens: list, dag: nx.DiGraph, target_path: str, max_ceiling: int = 1500, seed: int = 42):
    """Render dual-panel diagnostics figure: payload token histogram and sample taxonomic DAG."""
    os.makedirs(os.path.dirname(target_path), exist_ok=True)
    plt.figure(figsize=(14, 6))

    # Subplot 1: Payload Token Distribution
    plt.subplot(1, 2, 1)
    plt.hist(tokens, bins=15, color="steelblue", edgecolor="black", alpha=0.7)
    plt.axvline(max_ceiling, color="crimson", linestyle="--", linewidth=2, label=f"Max Ceiling ({max_ceiling})")
    plt.title("Token Payload Distribution", fontsize=12)
    plt.xlabel("Tokens per Payload", fontsize=10)
    plt.ylabel("Payload Count", fontsize=10)
    plt.legend()
    plt.grid(True, linestyle=":", alpha=0.6)

    # Subplot 2: Induced Subsumption Hierarchy DAG
    plt.subplot(1, 2, 2)
    sample_nodes = list(dag.nodes())[:15]
    sub_dag = dag.subgraph(sample_nodes)
    pos = nx.spring_layout(sub_dag, seed=seed)
    labels = {n: local_name(n) for n in sample_nodes}

    nx.draw_networkx_nodes(sub_dag, pos, node_size=600, node_color="skyblue", alpha=0.9)
    nx.draw_networkx_edges(sub_dag, pos, arrowstyle="->", arrowsize=12, edge_color="gray", width=1.5)
    nx.draw_networkx_labels(sub_dag, pos, labels=labels, font_size=8)
    plt.title("Sample Induced Taxonomic DAG (Minimal Hasse)", fontsize=12)
    plt.axis("off")

    plt.tight_layout()
    plt.savefig(target_path, dpi=150)
    plt.close()


telemetry_plot_path = os.environ.get("TELEMETRY_PLOT_PATH", os.path.join(BASE_DIR, "output", "telemetry_diagnostics.png"))
render_telemetry_dashboard(payload_tokens_list, G_reduced, telemetry_plot_path, CONFIG["MAX_PAYLOAD_TOKENS"])

print(f"Diagnostic plot saved to {telemetry_plot_path}.")
print("Pipeline completed successfully.")
